In [ ]:
import pandas as pd
from time import perf_counter

In [ ]:
import sys
import os

sys.path.append(os.path.abspath(os.path.join('..')))

In [ ]:
from models import ID3_DecisionTreeClassifier, NaiveBayesClassifier
from preprocessing import train_test_split, LabelEncoder, OrdinalEncoder
import visuals as vis

In [ ]:
df = pd.read_csv("../data/raw/mushroom_csv.csv")
df.head()

In [ ]:
vis.plot_missing_values(df.drop(['class'], axis=1))
df = df.fillna('None')

In [ ]:
Y = df['class'].values
X = df.drop(['class'], axis=1).values


vis.plot_class_distribution(Y)

In [ ]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y)

In [ ]:
le = LabelEncoder()
oe = OrdinalEncoder()

Y_train = le.fit_transform(Y_train)
Y_test  = le.transform(Y_test)

X_train = oe.fit_transform(X_train, Y_train)
X_test  = oe.transform(X_test)

In [ ]:
m1 = ID3_DecisionTreeClassifier(max_depth=2, criterion="entropy")
m2 = ID3_DecisionTreeClassifier(max_depth=2, criterion="gini")
m3 = NaiveBayesClassifier()


models = [m1, m2, m3]
fit_times = []

for m in models:
    start = perf_counter()
    m.fit(X_train, Y_train)
    end = perf_counter()

    fit_times.append(end - start)

In [ ]:
for i, m in enumerate(models):
    score = m.score(X_test, Y_test, onlyvalues=True)[0]
    print(f"model {i+1}: {round(score, 4)}, fitting time: {round(fit_times[i], 4)}")


In [ ]:
conf_mats = [m.score(X_test, Y_test, scores=["confusion_matrix"], onlyvalues=True)[0] for m in models]

vis.plot_confusion_matrices(
    conf_mats,
    ["edible", "poisonous"],
    titling=lambda i: f"Confusion Matrix for Model {i}"
)